In [1]:
%cd ../..

/home/eli/AnacondaProjects/epych


In [2]:
%env DASK_LOGGING__DISTRIBUTED=CRITICAL
%env OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
%env SPYTMPDIR=/mnt/data/tmp_storage
%env SPYLOGLEVEL=CRITICAL
%env SPYPARLOGLEVEL=CRITICAL

env: DASK_LOGGING__DISTRIBUTED=CRITICAL
env: OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
env: SPYTMPDIR=/mnt/data/tmp_storage
env: SPYLOGLEVEL=CRITICAL
env: SPYPARLOGLEVEL=CRITICAL


In [3]:
import collections
import glob
import functools
from hyppo.ksample import Hotelling
import logging
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import quantities as pq
import scipy.stats as stats

import epych
from epych.statistics import alignment

[striatum:519916] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.striatum.1000/jf.0/526778368/shared_mem_cuda_pool.striatum could be created.
[striatum:519916] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


In [4]:
%matplotlib inline

In [5]:
logging.basicConfig(level=logging.INFO)

In [6]:
CONDITIONS = ["go_gloexp", "go_seqctl", "lo_gloexp", "lonaive", "lo_rndctl", "igo_seqctl"]
PRETRIAL_SECONDS = 0.5
POSTTRIAL_SECONDS = 0.5
FORCE_RECOMPUTE = False
anatomical_areas = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

In [7]:
NWB_SUBJECTS = glob.glob('/mnt/data/000253/sub-*/')

In [8]:
PILOT_FILES = []

In [9]:
ODDBALL_ONSET = pq.Quantity(-1.9017372477960602e-14) * pq.second
ODDBALL_OFFSET = pq.Quantity(0.5004545430388676) * pq.second
OFFSET = pq.Quantity(0.020) * pq.second

In [10]:
aligner = epych.statistics.alignment.AlignmentSummary.unpickle("/mnt/data/000253/visual_alignment")
AREA_COUNTER = collections.Counter()

In [11]:
def visual_align(signal):
    visual = signal.select_channels(["VIS" in loc for loc in signal.channels.location]).median_filter()
    area = alignment.location_prefix(None, visual)
    result = aligner.stats[area].align(AREA_COUNTER[area], visual)
    AREA_COUNTER[area] += 1
    return result

In [12]:
def samplings(s, subject_dir, cond):
    subject = subject_dir.split('/')[-2]
    sampling = epych.recording.Sampling.unpickle(subject_dir + "/" + cond).smap(
        lambda sig: visual_align(sig)[(ODDBALL_ONSET - OFFSET).magnitude:(ODDBALL_OFFSET + OFFSET).magnitude]
    )
    logging.info("Loaded LFPs for %s in subject %s" % (cond, subject))
    yield sampling
    del sampling

In [13]:
def initialize_spectrum(key, signal):
    area = os.path.commonprefix([loc for loc in signal.channels.location])
    return epych.statistics.spectrum.PowerSpectrum(signal.df, signal.channels, signal.f0, taper="hann")

In [14]:
summaries = {cond: {} for cond in CONDITIONS}

In [15]:
for cond in CONDITIONS:
    global AREA_COUNTER
    AREA_COUNTER = collections.Counter()
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        if not os.path.exists(subject_dir + "/" + cond):
            continue
        if os.path.exists("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond)) and not FORCE_RECOMPUTE:
            summaries[cond][subject] = epych.statistic.Summary.unpickle("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond),
                                                                        epych.statistics.spectrum.PowerSpectrum)
        else:
            summaries[cond][subject] = epych.statistic.Summary(alignment.location_prefix, initialize_spectrum)
            summaries[cond][subject].calculate(samplings(s, subject_dir, cond))
            summaries[cond][subject].pickle("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond))
        logging.info("Analyzed spectra from %s LFPs in subject %s" % (cond, subject))

INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-621890
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-632485
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-632487
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637542
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637908
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637909
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-640507
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-642507
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-645322
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-645324
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-645495
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-647836
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-649323
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-649324
INFO:root:Analyzed spectra from go

In [16]:
aperiodic_params = {cond: {area: [] for area in anatomical_areas} for cond in CONDITIONS}

for cond in CONDITIONS:
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        for area in summaries[cond][subject].stats:
            if area not in anatomical_areas:
                continue
            aperiodic_params[cond][area].append(summaries[cond][subject].stats[area].aperiodic_parameters())
    for area in anatomical_areas:
        aperiodic_params[cond][area] = np.stack(aperiodic_params[cond][area], axis=0)

/home/eli/miniforge3/envs/ephys/lib/python3.12/site-packages/fooof/core/funcs.py:62: RuntimeWarning: invalid value encountered in log10
  ys = offset - np.log10(knee + xs**exp)
/home/eli/miniforge3/envs/ephys/lib/python3.12/site-packages/fooof/core/funcs.py:62: RuntimeWarning: invalid value encountered in log10
  ys = offset - np.log10(knee + xs**exp)


In [17]:
CONTRASTS = [("go_contrast", "go_gloexp", "go_seqctl"), ("ssa", "lo_gloexp", "igo_seqctl"), ("dd", "lo_rndctl", "lonaive")]
contrast_results = {name: {area: None for area in anatomical_areas} for (name, _, _) in CONTRASTS}

In [18]:
for name, condl, condr in CONTRASTS:
    for area in anatomical_areas:
        contrast_results[name][area] = Hotelling().test(aperiodic_params[condl][area], aperiodic_params[condr][area])

In [19]:
for name, _, _ in CONTRASTS:
    for area in anatomical_areas:
        logging.info("Hotelling test results for %s contrast in area %s: %s" % (name, area, str(contrast_results[name][area])))

INFO:root:Hotelling test results for go_contrast contrast in area VISp: KSampleTestOutput(stat=0.3660583098053628, pvalue=0.7781126654165358)
INFO:root:Hotelling test results for go_contrast contrast in area VISl: KSampleTestOutput(stat=0.04284143703032662, pvalue=0.9878728596601163)
INFO:root:Hotelling test results for go_contrast contrast in area VISrl: KSampleTestOutput(stat=0.047815910300037213, pvalue=0.9858035088114249)
INFO:root:Hotelling test results for go_contrast contrast in area VISal: KSampleTestOutput(stat=0.1660148411766842, pvalue=0.9178824748856513)
INFO:root:Hotelling test results for go_contrast contrast in area VISpm: KSampleTestOutput(stat=0.4162182167360487, pvalue=0.7429308955396895)
INFO:root:Hotelling test results for go_contrast contrast in area VISam: KSampleTestOutput(stat=0.2801293491309322, pvalue=0.8391865841695665)
INFO:root:Hotelling test results for ssa contrast in area VISp: KSampleTestOutput(stat=0.5997193300768412, pvalue=0.6214357532655941)
INFO:ro